In [0]:
# CELL 1 — Confirm Spark is working
print(spark.version)
print("✅ Spark is running on Serverless")

4.1.0
✅ Spark is running on Serverless


In [0]:
# CELL 2 — Install dbldatagen
%pip install dbldatagen

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CELL 3 — Restart Python so library loads properly
dbutils.library.restartPython()

In [0]:
# CELL 4 — Generate synthetic data using plain PySpark (Spark 4.1 compatible)
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random

# Create base dataframe with 50,000 rows
df = spark.range(50000).withColumnRenamed("id", "section_id")

# Add all columns using Spark functions
df = df \
    .withColumn("city",
        F.element_at(
            F.array(F.lit("Chennai"), F.lit("Mumbai"), F.lit("Delhi"),
                    F.lit("Bengaluru"), F.lit("Hyderabad"), F.lit("Kolkata")),
            (F.abs(F.hash(F.col("section_id"))) % 6 + 1).cast("int")
        )
    ) \
    .withColumn("latitude",  F.round(F.rand() * 27 + 8,  4)) \
    .withColumn("longitude", F.round(F.rand() * 29 + 68, 4)) \
    .withColumn("texture_depth_variance", F.round(F.rand() * 10, 2)) \
    .withColumn("bump_intensity",         F.round(F.rand() * 10, 2)) \
    .withColumn("crack_intensity",        F.round(F.rand() * 10, 2)) \
    .withColumn("rut_depth",              F.round(F.rand() * 10, 2)) \
    .withColumn("edge_roughness",         F.round(F.rand() * 10, 2)) \
    .withColumn("longitudinal_variance",  F.round(F.rand() * 10, 2)) \
    .withColumn("road_type",
        F.element_at(
            F.array(F.lit("highway"), F.lit("arterial"),
                    F.lit("collector"), F.lit("local")),
            (F.abs(F.hash(F.col("section_id") + 1)) % 4 + 1).cast("int")
        )
    ) \
    .withColumn("daily_traffic",    (F.rand() * 79500 + 500).cast("int")) \
    .withColumn("accidents_per_km", F.round(F.rand() * 12, 2)) \
    .withColumn("survey_year",
        F.element_at(
            F.array(F.lit(2021), F.lit(2022), F.lit(2023), F.lit(2024)),
            (F.abs(F.hash(F.col("section_id") + 2)) % 4 + 1).cast("int")
        )
    ) \
    .withColumn("pothole_detected",
        F.when(F.rand() < 0.05, 1).otherwise(0)
    ) \
    .withColumn("pothole_count",
        F.when(F.col("pothole_detected") == 1,
               (F.rand() * 7 + 1).cast("int")
        ).otherwise(0)
    )

print(f"✅ Generated {df.count():,} rows")
df.show(5)

✅ Generated 50,000 rows
+----------+---------+--------+---------+----------------------+--------------+---------------+---------+--------------+---------------------+---------+-------------+----------------+-----------+----------------+-------------+
|section_id|     city|latitude|longitude|texture_depth_variance|bump_intensity|crack_intensity|rut_depth|edge_roughness|longitudinal_variance|road_type|daily_traffic|accidents_per_km|survey_year|pothole_detected|pothole_count|
+----------+---------+--------+---------+----------------------+--------------+---------------+---------+--------------+---------------------+---------+-------------+----------------+-----------+----------------+-------------+
|         0|  Kolkata| 18.8528|   95.129|                  2.83|          3.91|           0.28|     5.45|           2.3|                  8.9|    local|        62403|            7.19|       2021|               0|            0|
|         1|   Mumbai| 21.6102|  82.9091|                  9.83|    

In [0]:
# CELL 5 — Save to Delta Lake
spark.sql("USE pothole_heatmap")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pothole_heatmap.road_sections_raw")

print("✅ Saved to Delta Lake!")

✅ Saved to Delta Lake!


In [0]:
# CELL 6 — Query with Spark SQL to verify
result = spark.sql("""
    SELECT 
        city,
        COUNT(*) as total_sections,
        ROUND(AVG(crack_intensity), 2) as avg_crack_score,
        ROUND(AVG(accidents_per_km), 2) as avg_accidents,
        SUM(pothole_detected) as potholes_found
    FROM pothole_heatmap.road_sections_raw
    GROUP BY city
    ORDER BY potholes_found DESC
""")

result.show()
print("✅ Delta Lake is working — data is queryable!")

+---------+--------------+---------------+-------------+--------------+
|     city|total_sections|avg_crack_score|avg_accidents|potholes_found|
+---------+--------------+---------------+-------------+--------------+
|   Mumbai|          8393|           5.01|         6.02|           453|
|  Kolkata|          8451|           4.97|         6.01|           451|
|  Chennai|          8291|           5.06|          6.1|           428|
|    Delhi|          8328|           5.02|         5.98|           418|
|Hyderabad|          8294|           4.99|         6.04|           381|
|Bengaluru|          8243|           4.94|         5.99|           356|
+---------+--------------+---------------+-------------+--------------+

✅ Delta Lake is working — data is queryable!
